# [실습] PDF 문서 요약

In [1]:
# Load environment variables in a file called .env
# API 키 설정 
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

# Check the key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

API key found and looks good so far!
Google API Key exists and begins AI


In [2]:
from langchain_community.document_loaders import PyMuPDFLoader
import re

# ── 1. PDF 로드 ──────────────────────────────────────────────
# PyMuPDFLoader: PDF를 페이지 단위로 읽어 Document 객체 리스트로 반환
# 각 Document는 page_content(텍스트)와 metadata(페이지 번호 등)를 포함
pdf_path = 'test.pdf'  # 분석할 PDF 파일 경로 (필요 시 변경)
loader = PyMuPDFLoader(pdf_path)
pages = loader.load()  # 전체 페이지를 리스트로 불러옴

# ── 2. 페이지 텍스트 전처리 ───────────────────────────────────
# PDF에서 추출한 텍스트는 불필요한 공백·특수문자가 많아 정제가 필요
for page in pages:
    page.page_content = re.sub(
        r'[^\w\s]',   # 정규식: 영문자·숫자·한글·공백을 제외한 모든 특수문자 대상
        '',           # 매칭된 특수문자를 빈 문자열로 교체 (= 제거)
        page.page_content
            .replace('\xa0', '')   # \xa0 : HTML 엔티티 &nbsp; (줄바꿈 없는 공백) 제거
            .replace('\n\n', '\n') # 연속된 빈 줄을 단일 줄바꿈으로 압축
    )

# ── 3. 특정 페이지 확인 ───────────────────────────────────────
# pages는 0-index 리스트이므로 pages[3]은 PDF의 4번째 페이지에 해당
print(pages[3])  # 전처리 결과 확인용 출력 (인덱스 변경으로 원하는 페이지 확인 가능)

page_content='LGFA PLAYING Rules 2022 
2 
 
 
경기장 
1 
여자 게일식 축구는 U14 경기부터 GAA 풀사이즈 
경기장에서 진행됩니다 U13 이하 등급의 경우 경기장 
크기가 축소될 수 있습니다 
2 
15 인제 미만의 경기일 경우 조직 위원회는 경기장의 크기 
득점 공간 경기 시간을 단축할 수 있습니다 
 
선수 
1 
임신 가능성이 있거나 뇌진탕 등을 앓고 있는 선수는 여자 
게일식 축구를 할 수 없습니다 만약 출전하면 스스로 
전적인 위험을 감수해야 하며 여자 게일식 축구 협회는 
그에 따른 결과에 어떤 책임도 지지 않습니다 
 
선수 복장 
1 
여자 게일식 축구의 복장은 저지 반바지 양말 
부츠입니다 선수는 여자 게일식 축구를 하는 동안 
쥬얼리 귀걸이 머리핀 또는 부상을 유발할 수 있는 기타 
아이템을 착용할 수 없습니다 인조 표면에서 진행되는 
경기의 경우 팀 내 색상이 통일되었다는 전제하에 
선수들은 다리 보호대를 착용할 수 있습니' metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2022-09-02T03:22:38+09:00', 'source': 'test.pdf', 'file_path': 'test.pdf', 'total_pages': 11, 'format': 'PDF 1.7', 'title': '', 'author': 'user', 'subject': '', 'keywords': '', 'moddate': '2022-09-02T03:22:38+09:00', 'trapped': '', 'modDate': "D:20220902032238+09'00'", 'creationDate': "D:20220902032238+09'00'", 'page': 3}


### 1. Stuff 방식 요약


In [7]:
# 1. Stuff 방식 요약
# Stuff 방식은 전체 문서를 한 번에 요약하는 방법입니다.

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ── 1. 코퍼스 생성 ────────────────────────────────────────────
# 여러 페이지의 내용을 하나의 Document 객체로 합침
corpus = Document(page_content='')

for page in pages:
    corpus.page_content += page.page_content + '\n'

# 연속된 줄바꿈을 단일 줄바꿈으로 정리
corpus.page_content = corpus.page_content.replace('\n\n', '\n')

# ── 2. 청크 분할 ──────────────────────────────────────────────
# chunk_size=2000 : 청크 1개의 최대 글자 수
# chunk_overlap=200 : 청크 간 겹치는 글자 수 (문맥 유지)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
chunks = text_splitter.split_documents([corpus])

# ── 3. 프롬프트 템플릿 정의 ───────────────────────────────────
stuff_prompt_template = """주어진 문서의 내용을 요약하세요.
요약은 1개의 문단으로 구성되며, 각 문단은 2개의 문장으로 작성하세요.
---
{text}
---
요약:"""

stuff_prompt = PromptTemplate.from_template(stuff_prompt_template)

# ── 4. LLM 및 체인 구성 (LCEL 방식) ──────────────────────────
# load_summarize_chain 대신 LCEL 파이프라인으로 직접 구현
# stuff 방식: 모든 청크의 텍스트를 하나로 합쳐 한 번에 요약
llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0.1, max_tokens=4096)

stuff_chain = stuff_prompt | llm | StrOutputParser()

# ── 5. 요약 실행 ──────────────────────────────────────────────
# 모든 청크의 page_content를 하나의 문자열로 합쳐 전달 (stuff 방식)
combined_text = "\n".join([chunk.page_content for chunk in chunks])
result = stuff_chain.invoke({"text": combined_text})

print(result)

2022년 여자 게일식 축구 경기 규칙은 경기장 크기, 선수 구성, 복장, 경기 시간, 교체 및 연장 규정 등을 상세히 규정하고 있으며, 선수 안전과 공정한 경기 진행을 위해 출혈 및 두부 손상 의심 시 조치와 보호장비 착용 의무도 포함되어 있습니다. 또한, 경기 중 공 다루기, 신체 접촉 금지, 공 소유 시간 제한 등 경기 규칙과 안전 수칙을 명확히 제시하여 선수와 관계자들이 규칙을 이해하고 준수할 수 있도록 하고 있습니다.


# 2. Refine 방식 요약


In [8]:
# 2. Refine 방식 요약
# 이 요약 방법은 긴 텍스트를 여러 부분(청크)으로 나눠 단계별로 요약하는 방식입니다.
# 먼저 첫 번째 청크를 요약하고, 그 후 나머지 청크들을 하나씩 추가하면서 기존 요약을 보완해 나갑니다.
# 최종적으로 모든 청크를 통합하여 완성된 요약이 출력됩니다.
# 요약은 처음 요약과 새로운 청크를 기반으로 반복적으로 업데이트됩니다.

# 초기 요약을 위한 프롬프트 템플릿을 정의합니다.
# 주어진 텍스트를 2개의 문단과 각 문단별 5개의 문장으로 요약하도록 지시합니다.
initial_prompt_template = """주어진 텍스트를 2개의 문단과 각 문단별 5개의 문장으로 요약하세요."""

# 첫 번째 청크에 대해 초기 요약을 수행합니다.
# 'chunks[0].page_content'는 첫 번째 청크의 내용을 의미하며, 이를 요약합니다.
initial_summary = llm.invoke(
    PromptTemplate.from_template(initial_prompt_template).format(text=chunks[0].page_content)
).content

# 추가적인 요약 보완을 위한 프롬프트 템플릿을 정의합니다.
# 현재까지 요약된 내용을 참고하여, 새로운 청크의 내용을 추가하고 요약을 보완하도록 지시합니다.
refine_prompt_template = """현재 요약된 내용을 참고하여 새로운 내용을 추가하고, 요약을 보완하세요.
---
현재 요약: {previous_summary}
새로운 내용: {new_text}"""

# 첫 번째 청크 이후의 나머지 청크들을 반복하여 처리합니다.
# 각 청크마다 현재까지의 요약을 업데이트합니다.
for i in range(1, len(chunks)):
    refine_response = llm.invoke(
        PromptTemplate.from_template(refine_prompt_template).format(
            previous_summary=initial_summary,
            new_text=chunks[i].page_content
        )
    )
    # 새로 보완된 요약을 'initial_summary'에 저장합니다.
    initial_summary = refine_response.content

# 최종 요약 결과를 출력합니다.
print(initial_summary)


현재 요약: 물론입니다! 요약할 텍스트를 제공해 주세요.

추가 및 보완 내용:
- 각 선수의 등번호는 팀 명단에 기재된 선수명과 일치해야 하며, 경기 중 일관되게 유지되어야 합니다.
- 팀 명단이 15명을 초과하는 경우, 별도 명시가 없으면 처음 15명이 실제 경기 출전 선수로 간주됩니다.
- 클럽, 카운티 또는 주 차원에서는 선발 선수 15명과 교체 선수 명단을 조직위원회 또는 관련 위원회에 제출하여 공개해야 하며, 경기 규칙에 따라 교체는 정해진 횟수 내에서 이루어져야 합니다.
- 무제한 교체가 허용되는 경우도 있으나, 대부분의 공식 경기에서는 정해진 교체 횟수(일반적으로 5회 또는 경기 종료 시까지 허용된 횟수)를 준수해야 합니다.
- 특히, 출혈 또는 두부 손상 의심 선수는 규정에 따라 즉시 경기장에서 퇴장하며, 별도 교체 선수로 교체되어야 하고, 경기 재개 시 특별 규정을 따라야 합니다.
- 경기 중단 시, 주심은 출혈이 멈췄는지, 선수의 복장(유니폼)이 환복 또는 세척되었는지, 혈액 또는 두부 부상 의심 선수가 경기장에서 나갔는지 반드시 확인해야 합니다.
- 부상 선수가 다른 등번호의 유니폼을 입고 경기를 재개할 경우, 주심은 새 번호를 인지하고 기록해야 하며, 출혈 또는 두부 손상 의심 선수의 교체는 경기 규칙에 엄격히 따라야 하며, 선수의 안전이 최우선임을 명시합니다.
- 경기 중단 시, 주심은 출혈이 멈췄는지, 부상 선수의 상태를 확인하고, 필요 시 선수의 복장 교체 여부를 점검하며, 부상 선수의 안전과 공정한 경기 진행을 위해 철저히 관리해야 합니다.

이와 함께, 경기 규칙에 따라 선수는 서 있는 자세에서 공을 집거나 차는 행위, 공을 잡거나 튕기는 방법, 이동 시 공의 소유권 유지, 신체 접촉 금지, 공을 던지지 않는 규정 등을 준수해야 하며, 고의적 신체 접촉과 어깨싸움은 허용되지 않습니다.  
이 규정들은 여자 게일식 축구 협회(Croke Park, Dublin)에서 제정한 것으로, 경기의 공정성과 선수 안전을 최우선으로 합니다.


# 3. Map-Reduce 방식 요약


In [9]:
# 3. Map-Reduce 방식 요약
# 문서를 여러 청크로 나눈 후, 각각의 청크를 요약(Map 과정)하고, 그 요약들을 합쳐 최종 요약을 생성합니다(Reduce 과정).
# 각 청크를 요약하기 위한 프롬프트 템플릿을 정의합니다.
# 텍스트를 1-3개의 문단과 각 문단별 5개의 문장으로 요약하도록 지시합니다.
map_prompt_template = """주어진 텍스트를 1-3개의 문단과 각 문단별 5개의 문장으로 요약하세요."""
map_prompt = PromptTemplate.from_template(map_prompt_template)
# Map 과정: 각 청크에 대해 개별적으로 요약을 생성합니다.
map_summaries = []  # 각 청크의 요약을 저장할 리스트입니다.
for chunk in chunks:
    # 각 청크에 대해 요약을 생성하고, 결과를 리스트에 추가합니다.
    map_response = llm.invoke(map_prompt.format(text=chunk.page_content))
    map_summaries.append(map_response.content)

# Reduce 과정: 생성된 모든 요약을 합쳐 최종 요약을 만듭니다.
# 여러 개의 요약을 바탕으로 전체 문서의 내용을 요약하도록 지시하는 프롬프트 템플릿을 정의합니다.
reduce_prompt_template = """주어진 요약들을 바탕으로 전체 내용을 요약하세요.
요약은 5개의 문단으로 구성되며, 각 문단은 4-8개의 문장으로 작성하세요."""
reduce_prompt = PromptTemplate.from_template(reduce_prompt_template)

# 앞서 생성된 개별 요약들을 하나의 텍스트로 합친 후, 최종 요약을 생성합니다.
reduce_summary = llm.invoke(reduce_prompt.format(text='\n---\n'.join(map_summaries)))

# 최종 요약 결과를 출력합니다.
print(reduce_summary.content)


물론입니다. 요약할 내용을 제공해 주시면, 요청하신 대로 5개 문단으로 구성된 전체 요약을 작성해 드리겠습니다.
